# Bi-Mamba zh↔vi — Notebook xuất biểu đồ báo cáo

Notebook **độc lập**, **chạy được end-to-end** trên Colab (Run All):

* Tự `git clone` repo + cài deps + (tuỳ chọn) cài Mamba CUDA fast-path.
* **Tự `prepare_data.py` + `train_tokenizer.py`** nếu chưa có (mục 3b) — không
  cần chạy thủ công thứ tự nào trước.
* Sinh **toàn bộ dataset stats** (train/valid/test, per-source counts,
  length distributions, token-length, fertility) ở mục 8b.

Các phần phụ thuộc vào dữ liệu của lần train có sẵn — tự động skip nếu thiếu:
* **Loss curves** (mục 5–6): cần `TRAIN_LOG` (paste stdout của ô Train ở mục 4).
* **Param breakdown / BLEU / qualitative** (mục 8, 9–12): cần checkpoint
  trong `RUN_DIR` (Drive hoặc local). Mục 7 (LR schedule) chỉ cần config nên
  luôn chạy được.

Tất cả figure → `reports/figures/*.png`, các bảng → `reports/*.csv`.


## 1. Mount Drive (tuỳ chọn) + clone repo

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Skipping drive mount:', e)


In [ ]:
import os
REPO_URL = 'https://github.com/ChauDucToan/NLP_DHM.git'
REPO_DIR = '/content/NLP_DHM'
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only || true
%cd $REPO_DIR
!pwd && ls


## 2. Cài đặt dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q matplotlib pandas
import sys; sys.path.insert(0, 'src')
import torch
print('torch:', torch.__version__,
      '| CUDA:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


## 3. Cấu hình đường dẫn

`RUN_DIR` là nơi chứa checkpoint. `TRAIN_LOG` là file text bạn lưu stdout của
ô Train (xem mục 4 dưới). Nếu đã copy `runs/bi_mamba_55m/` lên Drive, đổi
đường dẫn cho phù hợp.

In [ ]:
from pathlib import Path
import yaml

CONFIG_PATH = Path('configs/_colab.yaml') if Path('configs/_colab.yaml').exists() else Path('configs/bi_mamba_55m.yaml')
RUN_DIR     = Path('runs/bi_mamba_55m')
TOKEN_DIR   = Path('data/tokenizer')
TRAIN_LOG   = Path('reports/training_stdout.log')   # bạn paste stdout vào file này (xem mục 4)
FIG_DIR     = Path('reports/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_LOG.parent.mkdir(parents=True, exist_ok=True)

# Ưu tiên Drive nếu đã được copy lên đó.
DRIVE_RUN_DIR = Path('/content/drive/MyDrive/bi_mamba_55m')
if DRIVE_RUN_DIR.exists():
    print(f'Using Drive run dir: {DRIVE_RUN_DIR}')
    RUN_DIR = DRIVE_RUN_DIR
    if (DRIVE_RUN_DIR / 'tokenizer' / 'spm.model').exists():
        TOKEN_DIR = DRIVE_RUN_DIR / 'tokenizer'
    drive_log = DRIVE_RUN_DIR / 'training_stdout.log'
    if drive_log.exists():
        TRAIN_LOG = drive_log
        print(f'Using Drive log: {TRAIN_LOG}')

print(f'CONFIG    = {CONFIG_PATH}')
print(f'RUN_DIR   = {RUN_DIR}')
print(f'TOKEN_DIR = {TOKEN_DIR}')
print(f'TRAIN_LOG = {TRAIN_LOG}  (exists={TRAIN_LOG.exists()})')
print(f'FIG_DIR   = {FIG_DIR}')
print()
print('Files in run dir:')
for p in sorted(RUN_DIR.glob('*'))[:30]:
    if p.is_file():
        print(f'  {p.name:30s} {p.stat().st_size/1e6:.1f} MB')


## 3b. Tự động chuẩn bị dữ liệu + tokenizer (chạy 1 lần, có cache)

Nếu `data/processed/train.jsonl` hoặc `data/tokenizer/spm.model` chưa có, ô
này sẽ tự gọi `scripts/prepare_data.py` (tải OPUS theo `data.preset` của
config, lọc + cap + split) rồi `scripts/train_tokenizer.py` (huấn luyện
SentencePiece BPE). Chạy lại lần sau sẽ skip vì file đã có sẵn.

Mặc định preset là `small` (~225K cặp sạch, ~25 MB download). Bạn có thể
đổi sang `tiny` (~50K cặp, ~1 MB) nếu chỉ muốn smoke-test nhanh, hoặc
`medium` để dùng nguyên dữ liệu đã train.


In [ ]:
import subprocess, sys, yaml
from pathlib import Path

cfg_dict = yaml.safe_load(open(CONFIG_PATH))
data_cfg = cfg_dict['data']
DATA_PROC = Path(data_cfg.get('processed_dir', 'data/processed'))
DATA_RAW  = Path(data_cfg.get('raw_dir',       'data/raw'))
DATA_TOK  = Path(data_cfg.get('tokenizer_dir', 'data/tokenizer'))
DATA_PROC.mkdir(parents=True, exist_ok=True)

def _run(cmd):
    print('  $', ' '.join(cmd))
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-2500:] if len(p.stdout) > 2500 else p.stdout, end='')
    if p.returncode != 0:
        print('STDERR:', p.stderr[-1500:])
        raise SystemExit(f'command failed: {cmd}')

# Step 1: download + clean + split
need_prep = not (DATA_PROC / 'train.jsonl').exists()
if need_prep:
    print('=== prepare_data.py ===')
    _run(['python', 'scripts/prepare_data.py',
          '--config', str(CONFIG_PATH),
          '--preset', str(data_cfg.get('preset', 'small'))])
else:
    print(f'cache hit: {DATA_PROC/"train.jsonl"} ({(DATA_PROC/"train.jsonl").stat().st_size/1e6:.1f} MB)')

# Step 2: train SentencePiece BPE tokenizer
spm_path = DATA_TOK / 'spm.model'
if not spm_path.exists():
    print('\n=== train_tokenizer.py ===')
    _run(['python', 'scripts/train_tokenizer.py', '--config', str(CONFIG_PATH)])
else:
    print(f'cache hit: {spm_path} ({spm_path.stat().st_size/1e3:.0f} KB)')

# Update TOKEN_DIR if we just trained one and Drive doesn't override.
if not (TOKEN_DIR / 'spm.model').exists() and (DATA_TOK / 'spm.model').exists():
    TOKEN_DIR = DATA_TOK
print(f'\nTOKEN_DIR (final) = {TOKEN_DIR}')


## 4. Cung cấp stdout của ô Train

Có **2 cách** để có file `TRAIN_LOG`:

**Cách A — paste trực tiếp vào ô bên dưới** (nhanh, không cần Drive).
Mở notebook gốc → click chuột phải vào output của ô Train → *Copy output* →
dán vào ô bên dưới (giữa 2 dấu `'''`).

**Cách B — đọc từ file `.log`** (đã `wget`/`copy` lên Colab hoặc Drive).
Bỏ qua cell A, đảm bảo `TRAIN_LOG` ở cell trên trỏ đúng tới file.

In [ ]:
# === CÁCH A: PASTE STDOUT VÀO ĐÂY ===
# Để trống nếu bạn đã có file ở TRAIN_LOG (cách B).
PASTED_STDOUT = '''
# Ví dụ vài dòng (paste hết stdout của ô Train của bạn vào đây):
#
# step 2000 | val_loss 4.752 (best 4.752) | ema_val_loss 4.812 (best 4.812)
#   -> new best val_loss 4.7521 @ step 2000 (saved best.pt)
#   -> new best ema_val_loss 4.8123 @ step 2000 (saved best_ema.pt)
# step 4000 | val_loss 3.581 (best 3.581) | ema_val_loss 3.762 (best 3.762)
#   ...
'''

if PASTED_STDOUT.strip() and 'step' in PASTED_STDOUT and 'val_loss' in PASTED_STDOUT:
    TRAIN_LOG.write_text(PASTED_STDOUT)
    print(f'Wrote {len(PASTED_STDOUT):,} bytes to {TRAIN_LOG}')
else:
    print(f'Skipping — using existing file at {TRAIN_LOG} (exists={TRAIN_LOG.exists()})')


## 5. Parse log → DataFrame

Trainer in ra 2 dạng dòng:
* `step N | val_loss V.VVV (best B.BBB) | ema_val_loss V.VVV (best B.BBB)` — khi EMA bật.
* `step N | val_loss V.VVV (best B.BBB)` — khi EMA tắt.

Regex bên dưới bắt cả 2 dạng.

In [ ]:
import re
import pandas as pd

# Soft-skip: nếu chưa có log, df_eval rỗng → các cell phụ thuộc tự bỏ qua.
HAVE_LOG = TRAIN_LOG.exists() and TRAIN_LOG.stat().st_size > 0
if not HAVE_LOG:
    print(f'NOTE: Không có log ở {TRAIN_LOG} → các biểu đồ loss sẽ skip. '
          f'Quay lại mục 4 paste stdout nếu muốn vẽ loss curves.')

PAT = re.compile(
    r'step\s+(\d+)\s+\|\s+val_loss\s+([\d.]+)\s+\(best\s+([\d.]+)\)'
    r'(?:\s+\|\s+ema_val_loss\s+([\d.]+)\s+\(best\s+([\d.]+)\))?'
)

rows = []
if HAVE_LOG:
    for line in TRAIN_LOG.read_text(errors='replace').splitlines():
        m = PAT.search(line)
        if not m:
            continue
        d = {
            'step': int(m.group(1)),
            'val_loss': float(m.group(2)),
            'best_val_loss': float(m.group(3)),
        }
        if m.group(4):
            d['ema_val_loss']      = float(m.group(4))
            d['best_ema_val_loss'] = float(m.group(5))
        rows.append(d)

df_eval = pd.DataFrame(rows)
if len(df_eval):
    df_eval = df_eval.drop_duplicates(subset=['step']).sort_values('step').reset_index(drop=True)

print(f'Parsed {len(df_eval)} eval points from log.')
if len(df_eval):
    print(f'  Steps: {df_eval.step.min():,} → {df_eval.step.max():,}')
    print(f'  Best val_loss = {df_eval.best_val_loss.min():.4f} '
          f'@ step {int(df_eval.loc[df_eval.val_loss.idxmin(), "step"]):,}')
    if 'best_ema_val_loss' in df_eval:
        print(f'  Best ema_val_loss = {df_eval.best_ema_val_loss.min():.4f} '
              f'@ step {int(df_eval.loc[df_eval.ema_val_loss.idxmin(), "step"]):,}')
df_eval.tail() if len(df_eval) else df_eval


## 6. Loss curves (val + EMA-val)

In [ ]:
import matplotlib.pyplot as plt

if len(df_eval) == 0:
    print('SKIP — không có log để vẽ loss curves (xem mục 4).')
else:
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(df_eval.step, df_eval.val_loss, marker='o', ms=4, color='C0', label='val_loss')
    if 'ema_val_loss' in df_eval:
        ax.plot(df_eval.step, df_eval.ema_val_loss, marker='s', ms=4, color='C1', label='ema_val_loss')

    bs = int(df_eval.loc[df_eval.val_loss.idxmin(), 'step'])
    bv = float(df_eval.val_loss.min())
    ax.axvline(bs, color='C0', ls='--', lw=0.7, alpha=0.5)
    ax.annotate(f'best val_loss={bv:.3f}\n@ step {bs:,}',
                xy=(bs, bv), xytext=(8, 14), textcoords='offset points',
                fontsize=9, color='C0')
    if 'ema_val_loss' in df_eval:
        bs2 = int(df_eval.loc[df_eval.ema_val_loss.idxmin(), 'step'])
        bv2 = float(df_eval.ema_val_loss.min())
        ax.axvline(bs2, color='C1', ls='--', lw=0.7, alpha=0.5)
        ax.annotate(f'best ema={bv2:.3f}\n@ step {bs2:,}',
                    xy=(bs2, bv2), xytext=(8, -22), textcoords='offset points',
                    fontsize=9, color='C1')

    ax.set_xlabel('Training step')
    ax.set_ylabel('Cross-entropy loss')
    ax.set_title('Bi-Mamba zh↔vi — validation loss')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', frameon=True)
    fig.tight_layout()
    out = FIG_DIR / 'loss_curves.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    print(f'Saved: {out}')
    plt.show()


## 7. LR schedule (tái dựng giải tích)

Dùng cùng công thức `cosine_lr` của trainer + config thực tế. Không cần parse log.

In [ ]:
import math
import yaml
import numpy as np

cfg = yaml.safe_load(open(CONFIG_PATH))
tr  = cfg['train']

def cosine_lr(step, warmup, max_steps, base_lr, min_lr):
    if step < warmup:
        return base_lr * (step + 1) / max(1, warmup)
    p = (step - warmup) / max(1, max_steps - warmup)
    p = min(max(p, 0.0), 1.0)
    return min_lr + 0.5 * (base_lr - min_lr) * (1.0 + math.cos(math.pi * p))

# Chọn endpoint = step cuối quan sát được trong log (nếu có), nếu không = max_steps.
endpoint = int(df_eval.step.max()) if len(df_eval) else int(tr['max_steps'])
xs = np.linspace(0, endpoint, num=400, dtype=int)
ys = [cosine_lr(int(s), tr['warmup_steps'], tr['max_steps'], tr['lr'], tr['min_lr']) for s in xs]

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(xs, ys, color='C2', lw=1.4)
ax.axvspan(0, tr['warmup_steps'], color='C2', alpha=0.07, label=f"warmup ({tr['warmup_steps']:,})")
ax.set_yscale('log'); ax.set_xlabel('Training step'); ax.set_ylabel('Learning rate (log)')
ax.set_title(f"LR schedule — warmup→cosine  (base={tr['lr']:.0e}, min={tr['min_lr']:.0e}, max_steps={tr['max_steps']:,})")
ax.grid(True, which='both', alpha=0.3); ax.legend()
fig.tight_layout()
out = FIG_DIR / 'lr_schedule.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved: {out}')
plt.show()


## 8. Bảng số tham số của model

Phân loại theo embedding (chia sẻ với lm_head), encoder (Bi-Mamba × N),
decoder (Mamba + cross-attention × N), norms.

In [ ]:
import torch
from bi_mamba_mt.model import BiMambaTranslator, ModelConfig

ckpt_for_arch = None
for cand in ['best_ema.pt', 'avg_last5_ema.pt', 'best.pt', 'latest.pt']:
    if (RUN_DIR / cand).exists():
        ckpt_for_arch = RUN_DIR / cand; break

HAVE_CKPT = ckpt_for_arch is not None
if not HAVE_CKPT:
    print(f'SKIP — không tìm thấy checkpoint trong {RUN_DIR}. '
          f'Phần param breakdown / BLEU / qualitative sẽ tự bỏ qua. '
          f'Copy `runs/bi_mamba_55m/` (hoặc đặt RUN_DIR sang Drive) để bật.')
    breakdown = None
    mcfg = None
else:
    ckpt = torch.load(ckpt_for_arch, map_location='cpu', weights_only=False)
    mcfg = ModelConfig(**ckpt['model_cfg'])
    model = BiMambaTranslator(mcfg)

    def _sum(prefix):
        return sum(p.numel() for n, p in model.named_parameters() if n.startswith(prefix))

    stats = {
        'embedding (tied with lm_head)':    _sum('embedding.'),
        'encoder (Bi-Mamba × N)':            _sum('encoder_layers.'),
        'decoder (Mamba + cross-attn × N)':  _sum('decoder_layers.'),
        'final norms':                       _sum('encoder_norm.') + _sum('decoder_norm.'),
    }
    total = sum(stats.values())

    import pandas as pd
    breakdown = pd.DataFrame(
        [(k, v, 100*v/total) for k, v in stats.items()] + [('TOTAL', total, 100.0)],
        columns=['component', 'params', '% of total']
    )
    breakdown['params'] = breakdown['params'].map(lambda x: f'{x:,}')
    print(f'Architecture: d_model={mcfg.d_model}, encoder={mcfg.n_encoder_layers}, '
          f'decoder={mcfg.n_decoder_layers}, d_ff={mcfg.d_ff}, vocab={mcfg.vocab_size}')
    breakdown.to_csv('reports/param_breakdown.csv', index=False)
breakdown


## 8b. Thống kê dataset

Thống kê đầy đủ trên dữ liệu thực tế đã chuẩn bị xong (sau khi `prepare_data.py`
đã được chạy). Sử dụng:
* **`data/processed/{train,valid,test}.jsonl`** — file kết quả cuối cùng đã
  qua filter + cap + split. Đây là dữ liệu mà model được train trên đó.
* **`data/raw/<source>.zip`** — các archive OPUS đã download (nếu còn). Dùng
  để đếm số cặp gốc của TỪNG nguồn (TED2020 / WikiMatrix / bible-uedin / ...)
  trước khi lọc + cap.

Nếu bạn đã xoá `data/raw/` để tiết kiệm dung lượng, phần "per-source" sẽ tự
skip (chỉ thiếu 1 biểu đồ; phần còn lại vẫn chạy).


### 8b.1. Train / Valid / Test split

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PROC = Path('data/processed')

def _read_jsonl(p):
    if not p.exists():
        return []
    out = []
    with open(p, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line: out.append(json.loads(line))
    return out

splits = {'train': _read_jsonl(DATA_PROC/'train.jsonl'),
          'valid': _read_jsonl(DATA_PROC/'valid.jsonl'),
          'test':  _read_jsonl(DATA_PROC/'test.jsonl')}
sizes = {k: len(v) for k, v in splits.items()}
total = sum(sizes.values())
print(f'Total pairs: {total:,}')
for k, n in sizes.items():
    print(f'  {k:5s}: {n:>10,}  ({100*n/max(total,1):5.1f}%)')

fig, ax = plt.subplots(figsize=(7, 3.6))
ks = list(sizes.keys()); vs = [sizes[k] for k in ks]
bars = ax.bar(ks, vs, color=['C0','C2','C3'])
for b, v in zip(bars, vs):
    ax.text(b.get_x() + b.get_width()/2, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Số cặp song ngữ')
ax.set_title('Train / Valid / Test split')
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
out = FIG_DIR / 'split_sizes.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved: {out}')
plt.show()


### 8b.2. Số cặp theo nguồn (per-source)

Đếm trực tiếp từ archive OPUS trong `data/raw/*.zip`:
* **raw**: số cặp gốc trong file zip OPUS, không filter.
* **after_filter**: số cặp pass `pair_ok` + `looks_like_zh`/`looks_like_vi` +
  ratio bounds — đúng pipeline của `prepare_data.py`.
* **after_cap**: áp dụng tiếp `data.max_pairs_per_source` (cap mỗi nguồn).


In [ ]:
import io, zipfile, sys
sys.path.insert(0, 'src')
from bi_mamba_mt.data import basic_clean, pair_ok, looks_like_zh, looks_like_vi
import yaml

cfg = yaml.safe_load(open(CONFIG_PATH))
data_cfg = cfg['data']
RAW_DIR = Path(data_cfg.get('raw_dir', 'data/raw'))
caps = data_cfg.get('max_pairs_per_source') or {}

# (source_key, zip filename pattern, vi_member, zh_member)
SOURCES = [
    ('ted2020',     'TED2020',      '.vi-zh.vi',     '.vi-zh.zh'),
    ('wikimatrix',  'WikiMatrix',   '.vi-zh.vi',     '.vi-zh.zh'),
    ('bible_uedin', 'bible-uedin',  '.vi-zh.vi',     '.vi-zh.zh'),
    ('opensubtitles','OpenSubtitles','.vi-zh_cn.vi', '.vi-zh_cn.zh_cn'),
    ('nllb',        'NLLB',         '.vi-zh.vi',     '.vi-zh.zh'),
]

def _count(zip_path, vi_pat, zh_pat):
    raw = passed = 0
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        vi_m = next((n for n in names if n.endswith(vi_pat)), None)
        zh_m = next((n for n in names if n.endswith(zh_pat)), None)
        if not (vi_m and zh_m): return 0, 0
        with z.open(vi_m) as fv, z.open(zh_m) as fz:
            for vi_line, zh_line in zip(io.TextIOWrapper(fv, encoding='utf-8'),
                                         io.TextIOWrapper(fz, encoding='utf-8')):
                vi = basic_clean(vi_line); zh = basic_clean(zh_line)
                if not vi or not zh: continue
                raw += 1
                # Same filter pipeline as prepare_data.py
                if not pair_ok(zh=zh, vi=vi,
                               min_zh_vi_ratio=data_cfg.get('min_zh_vi_ratio', 0.10),
                               max_zh_vi_ratio=data_cfg.get('max_zh_vi_ratio', 1.20),
                               script_check=data_cfg.get('script_check', True),
                               min_len=data_cfg.get('min_len', 1),
                               max_chars=data_cfg.get('max_len', 250)):
                    continue
                passed += 1
    return raw, passed

rows = []
for key, zname, vi_pat, zh_pat in SOURCES:
    # find zip
    # Zips are saved by prepare_data.py as <key>.zip (lowercase).
    cand = []
    by_key = RAW_DIR / f'{key}.zip'
    if by_key.exists(): cand = [by_key]
    else:
        cand = [p for p in RAW_DIR.glob(f'*{zname}*') if p.suffix == '.zip']
    if not cand:
        rows.append({'source': key, 'zip': None, 'raw': None, 'after_filter': None,
                     'after_cap': None, 'cap': caps.get(key)})
        continue
    zp = cand[0]
    try:
        raw, passed = _count(zp, vi_pat, zh_pat)
    except Exception as e:
        print(f'  {key}: failed to count ({e})')
        raw, passed = None, None
    cap = caps.get(key)
    capped = passed if cap is None else (None if passed is None else min(passed, int(cap)))
    rows.append({'source': key, 'zip': zp.name, 'raw': raw, 'after_filter': passed,
                 'after_cap': capped, 'cap': cap})

dfs = pd.DataFrame(rows)
print(dfs.to_string(index=False))
dfs.to_csv('reports/per_source_counts.csv', index=False)

# Bar chart - chỉ các nguồn có data
plotable = dfs.dropna(subset=['after_filter']).copy()
if len(plotable) > 0:
    fig, ax = plt.subplots(figsize=(11, 4.5))
    import numpy as np
    x = np.arange(len(plotable)); w = 0.28
    ax.bar(x - w, plotable['raw'].fillna(0),           w, label='raw',           color='#bbb')
    ax.bar(x,     plotable['after_filter'].fillna(0),  w, label='after filter',  color='C0')
    ax.bar(x + w, plotable['after_cap'].fillna(0),     w, label='after cap',     color='C2')
    ax.set_xticks(x); ax.set_xticklabels(plotable['source'], rotation=15)
    ax.set_yscale('log')
    ax.set_ylabel('Số cặp (log)')
    ax.set_title('Số cặp theo nguồn — raw / after filter / after cap')
    ax.legend(); ax.grid(True, axis='y', which='both', alpha=0.3)
    for i, r in enumerate(plotable.itertuples()):
        for j, (col, off) in enumerate([('raw', -w), ('after_filter', 0), ('after_cap', w)]):
            v = getattr(r, col)
            if v is not None and v > 0:
                ax.text(i + off, v, f'{int(v):,}', ha='center', va='bottom', fontsize=7, rotation=90)
    fig.tight_layout()
    out = FIG_DIR / 'per_source_counts.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    print(f'Saved: {out}')
    plt.show()
else:
    print('Không có archive trong data/raw/ — bỏ qua biểu đồ per-source.')


### 8b.3. Phân phối độ dài câu (character-level, train set)

Histogram độ dài (số ký tự) của câu zh và câu vi trong tập train. Chỉ hiện
trong khoảng [0, 250] vì đây là `max_len` của filter.

In [ ]:
import numpy as np

train = splits['train']
zh_lens = np.array([len(p['zh']) for p in train])
vi_lens = np.array([len(p['vi']) for p in train])

def _stats(arr):
    return {
        'count': len(arr),
        'mean':  float(arr.mean()) if len(arr) else 0,
        'median': float(np.median(arr)) if len(arr) else 0,
        'p95':   float(np.percentile(arr, 95)) if len(arr) else 0,
        'max':   int(arr.max()) if len(arr) else 0,
    }
print('Char-length stats (train set):')
print('  zh:', _stats(zh_lens))
print('  vi:', _stats(vi_lens))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(zh_lens, bins=60, range=(0, 200), color='C0', alpha=0.85)
axes[0].set_xlabel('Số ký tự'); axes[0].set_ylabel('Số câu'); axes[0].set_title('Độ dài câu zh (train)')
axes[0].axvline(zh_lens.mean(), color='red', ls='--', lw=1, label=f'mean={zh_lens.mean():.1f}')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].hist(vi_lens, bins=60, range=(0, 200), color='C3', alpha=0.85)
axes[1].set_xlabel('Số ký tự'); axes[1].set_ylabel('Số câu'); axes[1].set_title('Độ dài câu vi (train)')
axes[1].axvline(vi_lens.mean(), color='red', ls='--', lw=1, label=f'mean={vi_lens.mean():.1f}')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

fig.tight_layout()
out = FIG_DIR / 'length_dist_chars.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved: {out}')
plt.show()


### 8b.4. Phân phối tỉ lệ độ dài zh/vi

Tỉ lệ `len(zh)/len(vi)` cho từng cặp. Empirically với OPUS-clean tỉ lệ này
nằm trong khoảng 0.20–0.60 (vi dài hơn zh do có khoảng trắng + dấu thanh).
Filter mặc định ở `prepare_data.py` cho qua khoảng 0.10–1.20.

In [ ]:
ratios = zh_lens / np.maximum(vi_lens, 1)
ratios = ratios[(ratios >= 0.0) & (ratios <= 2.5)]   # clip outliers cho đỡ vỡ chart
print(f'zh/vi length ratio — mean={ratios.mean():.3f}, median={np.median(ratios):.3f}, '
      f'p5={np.percentile(ratios, 5):.3f}, p95={np.percentile(ratios, 95):.3f}')

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.hist(ratios, bins=80, range=(0, 2), color='C4', alpha=0.85)
ax.axvline(ratios.mean(), color='red', ls='--', lw=1, label=f'mean={ratios.mean():.2f}')
ax.axvspan(data_cfg.get('min_zh_vi_ratio', 0.10), data_cfg.get('max_zh_vi_ratio', 1.20),
           color='green', alpha=0.07, label='filter range')
ax.set_xlabel('len(zh) / len(vi)'); ax.set_ylabel('Số câu')
ax.set_title('Phân phối tỉ lệ độ dài zh / vi (train)')
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
out = FIG_DIR / 'length_ratio.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved: {out}')
plt.show()


### 8b.5. Phân phối độ dài câu (token-level, sau SentencePiece)

Vì model nhận token chứ không phải ký tự, đây là histogram quan trọng nhất
khi đánh giá `max_src_len` / `max_tgt_len`. Random sample 5000 cặp cho nhanh
(đủ chính xác đại diện).

In [ ]:
from bi_mamba_mt.tokenizer import Tokenizer
import random

spm_path = TOKEN_DIR / 'spm.model'
if not spm_path.exists():
    print(f'SKIP — không tìm thấy {spm_path}.')
else:
    tok = Tokenizer(str(spm_path))
    print(f'Vocab size: {tok.vocab_size:,}')

    n_sample = min(5000, len(train))
    sample = random.Random(42).sample(train, n_sample)
    zh_tok_lens = np.array([len(tok.encode(p['zh'])) for p in sample])
    vi_tok_lens = np.array([len(tok.encode(p['vi'])) for p in sample])

    print(f'\nToken-length stats (sample {n_sample}):')
    print(f'  zh: mean={zh_tok_lens.mean():.1f}  median={np.median(zh_tok_lens):.0f}  p95={np.percentile(zh_tok_lens,95):.0f}  max={zh_tok_lens.max()}')
    print(f'  vi: mean={vi_tok_lens.mean():.1f}  median={np.median(vi_tok_lens):.0f}  p95={np.percentile(vi_tok_lens,95):.0f}  max={vi_tok_lens.max()}')
    print(f'\nFertility (chars / token):')
    print(f'  zh: {(np.array([len(p["zh"]) for p in sample])/np.maximum(zh_tok_lens,1)).mean():.2f}')
    print(f'  vi: {(np.array([len(p["vi"]) for p in sample])/np.maximum(vi_tok_lens,1)).mean():.2f}')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].hist(zh_tok_lens, bins=60, range=(0, 100), color='C0', alpha=0.85)
    axes[0].set_xlabel('Số token (SP)'); axes[0].set_ylabel('Số câu')
    axes[0].set_title(f'Token-length zh (sample n={n_sample})')
    axes[0].axvline(zh_tok_lens.mean(), color='red', ls='--', lw=1, label=f'mean={zh_tok_lens.mean():.1f}')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].hist(vi_tok_lens, bins=60, range=(0, 100), color='C3', alpha=0.85)
    axes[1].set_xlabel('Số token (SP)'); axes[1].set_ylabel('Số câu')
    axes[1].set_title(f'Token-length vi (sample n={n_sample})')
    axes[1].axvline(vi_tok_lens.mean(), color='red', ls='--', lw=1, label=f'mean={vi_tok_lens.mean():.1f}')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    out = FIG_DIR / 'length_dist_tokens.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    print(f'\nSaved: {out}')
    plt.show()


### 8b.6. Bảng tổng hợp dataset

Bảng tổng hợp các con số quan trọng — paste thẳng vào báo cáo.

In [ ]:
dataset_summary = {
    'train pairs':              sizes['train'],
    'valid pairs':              sizes['valid'],
    'test pairs':               sizes['test'],
    'total':                    sum(sizes.values()),
    'zh char-length mean':      f'{zh_lens.mean():.1f}',
    'zh char-length p95':       f'{np.percentile(zh_lens, 95):.0f}',
    'vi char-length mean':      f'{vi_lens.mean():.1f}',
    'vi char-length p95':       f'{np.percentile(vi_lens, 95):.0f}',
    'zh/vi ratio mean':         f'{ratios.mean():.3f}',
    'preset':                   data_cfg.get('preset', '?'),
    'max_pairs_per_source':     data_cfg.get('max_pairs_per_source'),
    'config max_train_pairs':   data_cfg.get('max_train_pairs'),
}
ds = pd.DataFrame([(k, v) for k, v in dataset_summary.items()],
                  columns=['metric', 'value'])
ds.to_csv('reports/dataset_summary.csv', index=False)
ds


## 9. BLEU + chrF cho từng checkpoint (+ ensemble decode)

Chạy `scripts/evaluate.py` cho mỗi checkpoint + 1 lần ensemble 3 checkpoint
mạnh nhất, parse stdout → bar chart. Default 1000 cặp test, mất ~5–15 phút
trên T4 / RTX 6000.

In [ ]:
import os, re, json, subprocess

if not HAVE_CKPT:
    print('SKIP — không có checkpoint, bỏ qua phần BLEU/chrF.')
    results = {}
else:
    RESULTS_PATH = Path('reports/eval_results.json')
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

    EVAL_N = 1000

    CKPTS = [
        ('latest',         RUN_DIR / 'latest.pt'),
        ('latest_ema',     RUN_DIR / 'latest_ema.pt'),
        ('best',           RUN_DIR / 'best.pt'),
        ('best_ema',       RUN_DIR / 'best_ema.pt'),
        ('avg_last5',      RUN_DIR / 'avg_last5.pt'),
        ('avg_last5_ema',  RUN_DIR / 'avg_last5_ema.pt'),
    ]
    CKPTS = [(n, p) for n, p in CKPTS if p.exists()]

    ENSEMBLE_NAMES = ['best_ema', 'avg_last5_ema', 'best']
    ensemble_paths = [p for n, p in CKPTS if n in ENSEMBLE_NAMES]
    do_ensemble = len(ensemble_paths) >= 2

    help_out = subprocess.run(['python', 'scripts/evaluate.py', '--help'],
                              capture_output=True, text=True).stdout
    SUPPORTS_ENSEMBLE = '--checkpoints' in help_out
    if not SUPPORTS_ENSEMBLE and do_ensemble:
        print('NOTE: scripts/evaluate.py chưa hỗ trợ --checkpoints (chỉ có ở PR Path A). '
              'Bỏ qua phần ensemble cho an toàn.')
        do_ensemble = False

    pat = re.compile(r'\[(\w+)\]\s+n=(\d+)\s+BLEU=([\d.]+)\s+chrF=([\d.]+)\s+\(beam=(\d+),\s*lp=([\d.]+)\)')

    def _run(label, args):
        print(f'\n=== {label} ===')
        cmd = ['python', 'scripts/evaluate.py', '--config', str(CONFIG_PATH),
               '--num-samples', str(EVAL_N)] + args
        print('  $', ' '.join(cmd))
        out = subprocess.run(cmd, capture_output=True, text=True)
        print(out.stdout[-1500:])
        if out.returncode != 0:
            print('STDERR:', out.stderr[-1000:])
        parsed = {}
        for m in pat.finditer(out.stdout):
            d, n, bleu, chrf, beam, lp = m.groups()
            parsed[d] = {'n': int(n), 'bleu': float(bleu), 'chrf': float(chrf),
                         'beam': int(beam), 'lp': float(lp)}
        return parsed

    results = {}
    for name, ckpt_path in CKPTS:
        results[name] = _run(name, ['--checkpoint', str(ckpt_path)])

    if do_ensemble:
        label = f'ensemble x{len(ensemble_paths)}'
        results[label] = _run(label, ['--checkpoints'] + [str(p) for p in ensemble_paths])

    RESULTS_PATH.write_text(json.dumps(results, indent=2, ensure_ascii=False))
    print(f'\nSaved: {RESULTS_PATH}')
results


## 10. Bar chart: BLEU + chrF theo checkpoint (cả 2 chiều)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

if not results:
    print('SKIP — chưa có results, bỏ qua bar chart.')
else:
    order = [k for k in [
        'latest', 'latest_ema',
        'best', 'best_ema',
        'avg_last5', 'avg_last5_ema',
    ] if k in results] + [k for k in results if k.startswith('ensemble')]

    zh2vi_bleu = [results[k].get('zh2vi', {}).get('bleu', None) for k in order]
    vi2zh_bleu = [results[k].get('vi2zh', {}).get('bleu', None) for k in order]
    zh2vi_chrf = [results[k].get('zh2vi', {}).get('chrf', None) for k in order]
    vi2zh_chrf = [results[k].get('vi2zh', {}).get('chrf', None) for k in order]

    x = np.arange(len(order)); w = 0.38
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

    ax = axes[0]
    ax.bar(x - w/2, zh2vi_bleu, w, label='zh→vi', color='C0')
    ax.bar(x + w/2, vi2zh_bleu, w, label='vi→zh', color='C3')
    for i, v in enumerate(zh2vi_bleu):
        if v is not None: ax.text(i - w/2, v + 0.4, f'{v:.1f}', ha='center', fontsize=8)
    for i, v in enumerate(vi2zh_bleu):
        if v is not None: ax.text(i + w/2, v + 0.4, f'{v:.1f}', ha='center', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(order, rotation=20, ha='right')
    ax.set_ylabel('BLEU'); ax.set_title('SacreBLEU (n=1000) — both directions')
    ax.grid(True, axis='y', alpha=0.3); ax.legend()

    ax = axes[1]
    ax.bar(x - w/2, zh2vi_chrf, w, label='zh→vi', color='C0')
    ax.bar(x + w/2, vi2zh_chrf, w, label='vi→zh', color='C3')
    for i, v in enumerate(zh2vi_chrf):
        if v is not None: ax.text(i - w/2, v + 0.4, f'{v:.1f}', ha='center', fontsize=8)
    for i, v in enumerate(vi2zh_chrf):
        if v is not None: ax.text(i + w/2, v + 0.4, f'{v:.1f}', ha='center', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(order, rotation=20, ha='right')
    ax.set_ylabel('chrF'); ax.set_title('chrF (n=1000) — both directions')
    ax.grid(True, axis='y', alpha=0.3); ax.legend()

    fig.tight_layout()
    out = FIG_DIR / 'bleu_chrf_per_ckpt.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    print(f'Saved: {out}')
    plt.show()


## 11. Bảng tóm tắt (CSV cho báo cáo)

In [ ]:
if not results:
    print('SKIP — chưa có results, bỏ qua summary.csv.')
    summary = None
else:
    rows = []
    for name in order:
        r = results.get(name, {})
        rows.append({
            'checkpoint':       name,
            'zh→vi BLEU':       r.get('zh2vi', {}).get('bleu'),
            'zh→vi chrF':       r.get('zh2vi', {}).get('chrf'),
            'vi→zh BLEU':       r.get('vi2zh', {}).get('bleu'),
            'vi→zh chrF':       r.get('vi2zh', {}).get('chrf'),
        })
    import pandas as pd
    summary = pd.DataFrame(rows)
    summary.to_csv('reports/summary.csv', index=False)
summary


## 12. Vài câu dịch mẫu (qualitative)

In [ ]:
import torch
from bi_mamba_mt.model import BiMambaTranslator, ModelConfig
from bi_mamba_mt.tokenizer import Tokenizer
from bi_mamba_mt.translate import translate_batch

if not HAVE_CKPT:
    print('SKIP — không có checkpoint, bỏ qua qualitative samples.')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    demo_pri = ['avg_last5_ema.pt', 'best_ema.pt', 'best.pt', 'latest.pt']
    demo_path = next((RUN_DIR / p for p in demo_pri if (RUN_DIR / p).exists()), None)
    print(f'Demo checkpoint: {demo_path}')
    ckpt = torch.load(demo_path, map_location='cpu', weights_only=False)
    m = BiMambaTranslator(ModelConfig(**ckpt['model_cfg'])).to(device)
    m.load_state_dict(ckpt['model'], strict=False); m.eval()
    tok = Tokenizer(str(TOKEN_DIR / 'spm.model'))

    demos_zh2vi = [
        '我们今天去吃越南河粉。',
        '人工智能正在改变世界。',
        '她每天早上六点起床去跑步。',
        '机器翻译还有很多需要改进的地方。',
    ]
    demos_vi2zh = [
        'Hôm nay trời rất đẹp.',
        'Trí tuệ nhân tạo đang thay đổi thế giới.',
        'Cô ấy đã đi ngủ từ rất sớm.',
        'Học máy giúp giải quyết nhiều vấn đề khó.',
    ]

    print('\n--- zh → vi ---')
    out_zh2vi = translate_batch(m, tok, demos_zh2vi, 'zh2vi', beam_size=6, length_penalty=1.20, device=device)
    for s_, t in zip(demos_zh2vi, out_zh2vi):
        print(f'  {s_}\n    -> {t}')

    print('\n--- vi → zh ---')
    out_vi2zh = translate_batch(m, tok, demos_vi2zh, 'vi2zh', beam_size=6, length_penalty=0.80, device=device)
    for s_, t in zip(demos_vi2zh, out_vi2zh):
        print(f'  {s_}\n    -> {t}')

    samples = {
        'zh2vi': [{'src': s_, 'hyp': t} for s_, t in zip(demos_zh2vi, out_zh2vi)],
        'vi2zh': [{'src': s_, 'hyp': t} for s_, t in zip(demos_vi2zh, out_vi2zh)],
        'checkpoint': str(demo_path.name),
    }
    Path('reports/qualitative_samples.json').write_text(json.dumps(samples, indent=2, ensure_ascii=False))
    print('\nSaved: reports/qualitative_samples.json')


## 13. Done — files cho báo cáo

```
reports/
├── figures/
│   ├── loss_curves.png
│   ├── lr_schedule.png
│   ├── split_sizes.png
│   ├── per_source_counts.png
│   ├── length_dist_chars.png
│   ├── length_ratio.png
│   ├── length_dist_tokens.png
│   └── bleu_chrf_per_ckpt.png
├── eval_results.json
├── param_breakdown.csv
├── per_source_counts.csv
├── dataset_summary.csv
├── summary.csv
└── qualitative_samples.json
```
